# ML Training Lab 2000-2026

## Cosa fa questo notebook

Questo notebook addestra i modelli ML Stock Lab su pannelli fattoriali storici 2000-2026, valida lo split temporale e salva modelli, metriche, predizioni e metadata nel formato consumato dalla piattaforma Streamlit.

## Input / Output

- Input: Database Finanziario, `output/ml_training_lab/tables/FactorUniversePanel.csv`, artifact ML/valuation esistenti.
- Output: `output/ml_training_lab/tables/MLTraining_*`, `output/ml_training_lab/models/*.pkl`, `output/ml_stock_lab/tables/MLStockLab_model_comparison.csv`.
- Opzionale: Ollama locale per generare una sintesi in italiano dei risultati.


In [ ]:
# Parameters / setup
from pathlib import Path
import os
import sys

START_YEAR = 2000
END_YEAR = 2026
TRAIN_END_YEAR = 2018
TEST_START_YEAR = 2019
MODELS = ["ols", "rf"]
MAX_ROWS = 0  # 0 = full panel
RUN_DATA_BOOTSTRAP = False
EXECUTE_PROVIDER_CALLS = False
USE_OLLAMA = False
OLLAMA_MODEL = "llama3.1"

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/investment-research-platform-pro/research_platform_definitive')
except Exception:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name != 'research_platform_definitive':
        candidates = [PROJECT_ROOT / 'research_platform_definitive', PROJECT_ROOT.parent / 'research_platform_definitive']
        PROJECT_ROOT = next((p for p in candidates if p.exists()), PROJECT_ROOT)

for candidate in [PROJECT_ROOT, PROJECT_ROOT / 'src']:
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

OUTPUT_ROOT = PROJECT_ROOT / 'output'
FINANCIAL_DB_ROOT = os.environ.get('FINANCIAL_DB_ROOT')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)


## 1. Bootstrap dati e pannello fattoriale

In produzione il backfill storico viene normalmente eseguito da CLI o Notebook Runner. Questa cella resta disattivata di default per evitare download lunghi non intenzionali in Colab.


In [ ]:
from research_platform_core.data_completion import run_full_data_completion, validate_completion_coverage

if RUN_DATA_BOOTSTRAP:
    completion = run_full_data_completion(
        financial_db_root=FINANCIAL_DB_ROOT,
        output_root=OUTPUT_ROOT,
        start_year=START_YEAR,
        end_year=END_YEAR,
        execute=EXECUTE_PROVIDER_CALLS,
        max_assets=250,
        max_symbols=25,
    )
    print({name: len(frame) for name, frame in completion.items()})

coverage = validate_completion_coverage(FINANCIAL_DB_ROOT, OUTPUT_ROOT, START_YEAR, END_YEAR, strict=False)
coverage


## 2. Training ML Stock Lab

Il training usa split temporali per evitare look-ahead bias: default train 2000-2018 e test 2019-2026. Le metriche principali sono `R2_OS`, Sharpe long-short dei ranking e copertura del pannello.


In [ ]:
from ml_stock_lab.training import train_ml_model_suite

result = train_ml_model_suite(
    output_root=OUTPUT_ROOT,
    financial_db_root=FINANCIAL_DB_ROOT,
    start_year=START_YEAR,
    end_year=END_YEAR,
    train_end_year=TRAIN_END_YEAR,
    test_start_year=TEST_START_YEAR,
    models=MODELS,
    max_rows=None if MAX_ROWS == 0 else MAX_ROWS,
    use_ollama=USE_OLLAMA,
    ollama_model=OLLAMA_MODEL,
)
print(result['status'])
result['metrics']


## 3. Diagnostica predizioni e feature

Questa sezione controlla se le predizioni out-of-sample sono disponibili e quali feature guidano i modelli. La lettura resta descrittiva: un buon training richiede robustezza per periodo, settore e regime di mercato.


In [ ]:
display(result['coverage'])
display(result['feature_importance'].head(25))
display(result['predictions'].head(25))
print(result['paths'])


## 4. Ollama summary

Se `USE_OLLAMA=True` e Ollama e' raggiungibile, il notebook salva una sintesi in `output/ml_training_lab/MLTraining_ollama_summary.json`. La sintesi non sostituisce la validazione quantitativa: serve per accelerare review e documentazione.


In [ ]:
result['ollama_summary']


## Limiti

- La copertura storica reale dipende dai provider e dalle licenze disponibili.
- Il pannello fattoriale derivato da prezzi non sostituisce fondamentali point-in-time completi.
- Le metriche devono essere lette con controlli di leakage, turnover, costi e multiple testing.
- Prima di una release v1.0 usare `scripts/validate_research_data_coverage.py --strict`.
